In [1]:
import pandas as pd
dataset = pd.read_csv('CKD.csv')

In [2]:
dataset.shape

(399, 25)

In [3]:
dataset.head(5)

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [4]:
dataset=pd.get_dummies(dataset, dtype=int, drop_first=True)

In [5]:
dataset.shape

(399, 28)

In [6]:
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [9]:
independent=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
dependent=dataset[['classification_yes']]

In [10]:
dependent.value_counts()

classification_yes
1                     249
0                     150
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(independent,dependent,test_size=1/3,random_state=0)

In [13]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [15]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')
param_grid={'kernel':['linear', 'rbf', 'sigmoid'],
           'gamma':['scale','auto'],
           'C':[10,100,1000,2000,3000]}
grid = GridSearchCV(SVC(probability=True),param_grid,refit=True,n_jobs=-1,verbose=3,scoring='f1_weighted')

In [16]:
grid.fit(X_train,y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,SVC(probability=True)
,param_grid,"{'C': [10, 100, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf', ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [17]:
y_pred = grid.predict(X_test)

In [18]:
grid_result = grid.cv_results_

In [24]:
table = pd.DataFrame.from_dict(grid_result)

In [25]:
print(table)

    mean_fit_time  std_fit_time  mean_score_time  std_score_time  param_C  \
0        0.018361      0.005661         0.030691        0.006239       10   
1        0.022018      0.002974         0.033116        0.005033       10   
2        0.019783      0.003137         0.030751        0.007853       10   
3        0.016184      0.002334         0.032432        0.003056       10   
4        0.022696      0.004637         0.032137        0.004017       10   
5        0.018727      0.001800         0.027552        0.002172       10   
6        0.016439      0.003958         0.026258        0.003638      100   
7        0.018125      0.002155         0.027310        0.002323      100   
8        0.016731      0.002436         0.028917        0.001791      100   
9        0.014810      0.001764         0.026103        0.001144      100   
10       0.016493      0.001995         0.030206        0.003294      100   
11       0.016040      0.002083         0.025183        0.001706      100   

In [26]:
grid.best_params_

{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}

In [27]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score

In [29]:
cm=confusion_matrix(y_test,y_pred)
print(cm)

[[51  0]
 [ 1 81]]


Type1 error is 0 in the above confusion_matrix. And Type2 error is also 1. According to the above result this model is good. 

In [30]:
clf=classification_report(y_test,y_pred)
print(clf)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        51
           1       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



Accuracy on the above result is 99% which is too good. 

In [31]:
f1score = f1_score(y_test,y_pred)
print(f1score)

0.9938650306748467


F1score value is closer to 100% which is very good.

In [32]:
roc_auc_score1 = roc_auc_score(y_test, grid.predict_proba(X_test)[:,1])
print(roc_auc_score1)

1.0


In [33]:
import pickle
filename = "Finalized_Assignment_classifier.sav"
pickle.dump(grid,open(filename,'wb'))

In [45]:
train_columns = independent.columns
pickle.dump(train_columns, open('columns.sav', 'wb'))

In [34]:
loaded_model = pickle.load(open("Finalized_Assignment_classifier.sav",'rb'))

In [47]:
loaded_columns = pickle.load(open("columns.sav",'rb'))

In [48]:
sample_input = {
    'age': 48,
    'bp': 80,
    'sg': 'a',
    'al': 1,
    'su': 0,
    'rbc': 'normal',
    'pc': 'normal',
    'pcc': 'notpresent',
    'ba': 'notpresent',
    'bgr': 121,
    'bu': 36,
    'sc': 1.2,
    'sod': 135,
    'pot': 4.5,
    'hemo': 15.4,
    'pcv': 44,
    'wc': 7800,
    'rc': 5.2,
    'htn': 'yes',
    'dm': 'no',
    'cad': 'no',
    'appet': 'good',
    'pe': 'no',
    'ane': 'no'
}

In [49]:
input_df=pd.DataFrame([sample_input])

In [50]:
input_param=pd.get_dummies(input_df,dtype=int,drop_first=True)

In [51]:
input_param = input_param.reindex(columns=train_columns, fill_value=0)

In [52]:
loaded_model.predict(input_param)

array([0])